# 01b · Motor de enlaces — CNN de caracteres (deep learning)

**Fase 2 (parte profunda) — corre en Google Colab con GPU.**

El PDF (Paso 3 del motor de enlaces): *entrena una red convolucional sobre la secuencia
de caracteres de la URL*. Este es el modelo profundo que se compara contra el baseline
clásico (`01_url.ipynb`).

Idea: una URL es una secuencia de caracteres. La CNN aprende **subcadenas sospechosas**
(patrones locales) sin que nosotros las definamos a mano.

**Entregable:** modelo CNN guardado y sus predicciones sobre validación (para la Fase 4).

> ⚠️ En Colab: **Entorno de ejecución → Cambiar tipo → GPU (T4)** antes de entrenar.

In [ ]:
# En Colab: sube malicious_phish.csv (icono de carpeta → subir) o usa la API de Kaggle.
# Si no hay datos, el cuaderno usa un DEMO para probar el flujo.
import torch
print('GPU disponible:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', DEVICE)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MAX_FILAS = 80_000     # con GPU puedes subirlo o poner None para todo el dataset
MAX_LEN = 200          # longitud fija de la URL en caracteres
np.random.seed(SEED); torch.manual_seed(SEED)

def a_binario(v):
    s = str(v).strip().lower()
    return 'legitimo' if s in ('benign', 'legitimate', 'good', '0', 'legitimo') else 'fraudulento'

candidatos = [Path('malicious_phish.csv'),
              Path('Modelotest/data/crudo/malicious_phish.csv'),
              Path.cwd().parent / 'data' / 'crudo' / 'malicious_phish.csv']
ruta = next((p for p in candidatos if p.exists()), None)

if ruta is not None:
    print('Dataset real:', ruta)
    df = pd.read_csv(ruta)
    col_url = next((c for c in df.columns if c.lower() in ('url', 'urls', 'link')), df.columns[0])
    col_lab = next((c for c in df.columns if c.lower() in ('type', 'label', 'result', 'class')), df.columns[-1])
    df = df[[col_url, col_lab]].rename(columns={col_url: 'url', col_lab: 'lab'})
    df['etiqueta'] = df['lab'].map(a_binario)
else:
    print('Sin dataset -> DEMO (placeholder).')
    legit = [f'https://www.sitio{i}.com/pagina/articulo-{i}' for i in range(1500)]
    fraude = [f'http://secure-login-banco{i}.xyz/verify?acc={i}' for i in range(1500)]
    df = pd.DataFrame({'url': legit + fraude, 'etiqueta': ['legitimo']*1500 + ['fraudulento']*1500})

df = df.dropna(subset=['url']).drop_duplicates(subset=['url']).reset_index(drop=True)
print('Total:', len(df))
print(df['etiqueta'].value_counts().to_dict())

In [ ]:
from sklearn.model_selection import train_test_split

if MAX_FILAS and len(df) > MAX_FILAS:
    df, _ = train_test_split(df, train_size=MAX_FILAS, stratify=df['etiqueta'], random_state=SEED)
    df = df.reset_index(drop=True)

train, resto = train_test_split(df, test_size=0.30, stratify=df['etiqueta'], random_state=SEED)
val, test = train_test_split(resto, test_size=0.50, stratify=resto['etiqueta'], random_state=SEED)
print(f'train={len(train)}  val={len(val)}  test={len(test)}')

## Codificación a nivel de carácter

In [ ]:
# Vocabulario de caracteres a partir del train (0 = relleno/desconocido)
chars = sorted({c for u in train['url'] for c in u})
char2idx = {c: i + 1 for i, c in enumerate(chars)}
VOCAB = len(char2idx) + 1
print('Tamaño del vocabulario de caracteres:', VOCAB)

def codificar(url):
    v = [char2idx.get(c, 0) for c in str(url)[:MAX_LEN]]
    return v + [0] * (MAX_LEN - len(v))

lab2id = {'legitimo': 0, 'fraudulento': 1}

def a_tensores(part):
    X = torch.tensor([codificar(u) for u in part['url']], dtype=torch.long)
    y = torch.tensor([lab2id[e] for e in part['etiqueta']], dtype=torch.long)
    return X, y

X_tr, y_tr = a_tensores(train)
X_val, y_val = a_tensores(val)
print('X_tr:', tuple(X_tr.shape))

## La red convolucional

In [ ]:
import torch.nn as nn

class CharCNN(nn.Module):
    def __init__(self, vocab, emb_dim=32, n_clases=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(emb_dim, 128, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=5, padding=2)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.clasificador = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),  # dropout contra el sobreajuste
            nn.Linear(64, n_clases))

    def forward(self, x):
        x = self.emb(x).transpose(1, 2)      # (B, emb, L)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)         # (B, 128)
        return self.clasificador(x)

modelo = CharCNN(VOCAB).to(DEVICE)
print(modelo)

## Entrenamiento (con pesos por clase para el desbalance)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter

EPOCAS = 5
BATCH = 256

cnt = Counter(train['etiqueta']); total = sum(cnt.values())
pesos = torch.tensor([total / (2 * cnt[l]) for l in ['legitimo', 'fraudulento']],
                     dtype=torch.float).to(DEVICE)
criterio = nn.CrossEntropyLoss(weight=pesos)
optim = torch.optim.Adam(modelo.parameters(), lr=1e-3)

cargador = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)

for ep in range(1, EPOCAS + 1):
    modelo.train(); perdida_tot = 0.0
    for xb, yb in cargador:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optim.zero_grad()
        loss = criterio(modelo(xb), yb)
        loss.backward(); optim.step()
        perdida_tot += loss.item() * len(xb)
    print(f'época {ep}/{EPOCAS}  pérdida media = {perdida_tot/len(X_tr):.4f}')

## Evaluación en validación

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

modelo.eval()
with torch.no_grad():
    logits = modelo(X_val.to(DEVICE)).cpu()
pred_idx = logits.argmax(1).numpy()
id2lab = {0: 'legitimo', 1: 'fraudulento'}
pred = [id2lab[i] for i in pred_idx]
y_val_lab = val['etiqueta'].values

f1 = f1_score(y_val_lab, pred, pos_label='fraudulento', zero_division=0)
print(f'CNN de URL — F1 (fraudulento) = {f1:.3f}\n')
print(classification_report(y_val_lab, pred, digits=3, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

clases = ['legitimo', 'fraudulento']
cm = confusion_matrix(y_val_lab, pred, labels=clases)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=clases, yticklabels=clases, cbar=False, ax=ax)
ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real')
ax.set_title('CNN de URL (validación)')
plt.tight_layout(); plt.show()

In [ ]:
# Guardar modelo y predicciones (para la Fase 4). Ajusta las rutas si estás en Colab.
import json
torch.save(modelo.state_dict(), 'url_cnn.pt')
with open('url_cnn_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'char2idx': char2idx, 'max_len': MAX_LEN}, f)
pd.DataFrame({'url': val['url'].values, 'y_real': y_val_lab, 'pred_cnn': pred}).to_csv(
    'url_cnn_val_pred.csv', index=False)
print('Guardados url_cnn.pt, url_cnn_vocab.json y url_cnn_val_pred.csv ✔')
print('En Colab, copia estos archivos a tu Google Drive.')

## Qué sigue

- Compara este **F1 de la CNN** con el del baseline (`01_url.ipynb`).
- El veredicto formal (¿la CNN le gana al baseline?) se decide en `03_evaluacion.ipynb`
  con la **prueba de McNemar** sobre el conjunto de **test**, que sigue sin abrirse.
- Si sobreajusta (F1 alto en train, bajo en val): sube el `Dropout`, baja épocas o usa más datos.